In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, ArrayType,IntegerType, FloatType

# Create Spark session
spark = SparkSession.builder \
    .appName("Spark with Hive") \
    .enableHiveSupport() \
    .getOrCreate()

26/05/04 06:12:31 INFO SparkEnv: Registering MapOutputTracker
26/05/04 06:12:31 INFO SparkEnv: Registering BlockManagerMaster
26/05/04 06:12:31 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/04 06:12:31 INFO SparkEnv: Registering OutputCommitCoordinator


In [11]:
#Reading Movie data from HDFS path 
hdfs_path = '/tmp/data/assign2/movies.csv'
df_movies = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(hdfs_path)

# Print schema and sample data
df_movies.printSchema()
df_movies.show(5)

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



In [12]:
# Define the correct schema based on your CSV structure
schema_ratings = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp",IntegerType(), True),
])

#Reading ratings data from HDFS path 
hdfs_path = '/tmp/data/assign2/ratings.csv'
df_ratings = spark.read.format("csv").option('header','true').option('inferschema','true').schema(schema_ratings).load(hdfs_path)

#Enforce Timestamp column (Cast - Timestamp)
df_ratings= df_ratings.withColumn("timestamp",F.col("timestamp").cast("timestamp"))

#creating new column -year - extracting from Timestamp
df_ratings= df_ratings.withColumn("year", F.year("timestamp"))

# Print schema and sample data
df_ratings.printSchema
df_ratings.show(5)

+------+-------+------+-------------------+----+
|userId|movieId|rating|          timestamp|year|
+------+-------+------+-------------------+----+
|     1|      1|   4.0|2000-07-30 18:45:03|2000|
|     1|      3|   4.0|2000-07-30 18:20:47|2000|
|     1|      6|   4.0|2000-07-30 18:37:04|2000|
|     1|     47|   5.0|2000-07-30 19:03:35|2000|
|     1|     50|   5.0|2000-07-30 18:48:51|2000|
+------+-------+------+-------------------+----+
only showing top 5 rows



In [13]:
#Reading tags data from HDFS path 
hdfs_path = '/tmp/data/assign2/tags.csv'

# Define the correct schema based on your CSV structure
schema_tags =  StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("tag", StringType(), True),
    StructField("timestamp",IntegerType(), True),
])

#Reading tags data from HDFS path 
df_tags =spark.read.format("csv").option('header','true').option('inferschema','true').schema(schema_tags).load(hdfs_path)

#Enforce Timestamp column (Cast - Timestamp)
df_tags= df_tags.withColumn("timestamp", F.col("timestamp").cast("timestamp"))

# Print schema and sample data
df_tags.printSchema()
df_tags.show()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



+------+-------+-----------------+-------------------+
|userId|movieId|              tag|          timestamp|
+------+-------+-----------------+-------------------+
|     2|  60756|            funny|2015-10-24 19:29:54|
|     2|  60756|  Highly quotable|2015-10-24 19:29:56|
|     2|  60756|     will ferrell|2015-10-24 19:29:52|
|     2|  89774|     Boxing story|2015-10-24 19:33:27|
|     2|  89774|              MMA|2015-10-24 19:33:20|
|     2|  89774|        Tom Hardy|2015-10-24 19:33:25|
|     2| 106782|            drugs|2015-10-24 19:30:54|
|     2| 106782|Leonardo DiCaprio|2015-10-24 19:30:51|
|     2| 106782|  Martin Scorsese|2015-10-24 19:30:56|
|     7|  48516|     way too long|2007-01-25 01:08:45|
|    18|    431|        Al Pacino|2016-05-01 21:39:25|
|    18|    431|         gangster|2016-05-01 21:39:09|
|    18|    431|            mafia|2016-05-01 21:39:15|
|    18|   1221|        Al Pacino|2016-04-26 19:35:06|
|    18|   1221|            Mafia|2016-04-26 19:35:03|
|    18|  

In [14]:
#Create or Replace Temp View for SQL Queries
df_movies.createOrReplaceTempView("MOVIES")
df_ratings.createOrReplaceTempView("RATINGS")
df_tags.createOrReplaceTempView("TAGS")

In [15]:
#Que - Show the aggregated number of ratings per year
query ="""SELECT year , COUNT(rating) AS total_rating
            FROM RATINGS
            GROUP BY year
            ORDER BY year DESC"""

output = spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format('csv').option('header', 'true') .option('delimiter', ',').save('/tmp/output/assign2/agg_Ratings.csv')
print("Write Successfull")

+----+------------+
|year|total_rating|
+----+------------+
|2018|        6418|
|2017|        8198|
|2016|        6703|
|2015|        6616|
|2014|        1439|
|2013|        1664|
|2012|        4656|
|2011|        1690|
|2010|        2301|
|2009|        4158|
|2008|        4351|
|2007|        7114|
|2006|        4059|
|2005|        5813|
|2004|        3279|
|2003|        4014|
|2002|        3478|
|2001|        3922|
|2000|       10061|
|1999|        2439|
+----+------------+
only showing top 20 rows



Write Successfull


In [16]:
#Que - Show the average monthly number of ratings - For yyyy-mm format include date_format
query ="""SELECT  date_format(timestamp, 'yyyy-MM') AS year_month, avg(rating) AS avg_rating
            FROM RATINGS
            GROUP BY date_format(timestamp, 'yyyy-MM')
            ORDER BY year_month DESC"""

output = spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimitor',',').save('/tmp/output/assign2/avg_Ratings.csv')
print("successfull")

+----------+------------------+
|year_month|        avg_rating|
+----------+------------------+
|   2018-09| 3.568708609271523|
|   2018-08|3.5577617328519855|
|   2018-07| 4.010238907849829|
|   2018-06| 3.979713603818616|
|   2018-05|2.9516298633017874|
|   2018-04|              3.75|
|   2018-03| 3.786817713697219|
|   2018-02|2.7386655260906756|
|   2018-01|3.4194736842105264|
|   2017-12|3.2611940298507465|
|   2017-11| 3.652173913043478|
|   2017-10|3.5244444444444443|
|   2017-09|3.6827830188679247|
|   2017-08| 4.076923076923077|
|   2017-07| 4.052941176470588|
|   2017-06|2.9594240837696337|
|   2017-05| 3.480183562786817|
|   2017-04| 3.626218851570964|
|   2017-03| 3.051001821493625|
|   2017-02|2.7547619047619047|
+----------+------------------+
only showing top 20 rows



successfull


In [17]:
#Que- Show the rating levels distribution
query="""SELECT 
        CASE 
        WHEN rating >= 0.5 AND rating < 1.5 THEN '0.5–1.5'
        WHEN rating >= 1.5 AND rating < 2.5 THEN '1.5–2.5'
        WHEN rating >= 2.5 AND rating < 3.5 THEN '2.5–3.5'
        WHEN rating >= 3.5 AND rating < 4.5 THEN '3.5–4.5'
        WHEN rating >= 4.5 AND rating <= 5.0 THEN '4.5–5.0'
    END AS rating_bucket,
    COUNT(*) AS count
FROM RATINGS
GROUP BY rating_bucket
ORDER BY rating_bucket"""

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimitor',',').save('/tmp/output/assign2/distibuted_Ratings.csv')
print("successfull")

+-------------+-----+
|rating_bucket|count|
+-------------+-----+
|      0.5–1.5| 4181|
|      1.5–2.5| 9342|
|      2.5–3.5|25597|
|      3.5–4.5|39954|
|      4.5–5.0|21762|
+-------------+-----+



successfull


In [18]:
#Que - Show the 18 movies that are tagged but not rated

query="""WITH 
    tagged_movies AS (SELECT DISTINCT movieId FROM TAGS ), 
    rated_movies AS (SELECT DISTINCT movieId FROM RATINGS), 
    t1 AS ( SELECT movieId FROM tagged_movies WHERE movieId NOT IN (SELECT movieId FROM rated_movies))
    SELECT m.title
    FROM MOVIES m
    JOIN t1 ON m.movieId = t1.movieId
    ORDER BY m.title;"""

output= spark.sql(query)
output.show(18)

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimitor',',').save('/tmp/output/assign2/tagged_NotRated.csv')
print("successfull")

+--------------------+
|               title|
+--------------------+
|Browning Version,...|
|Call Northside 77...|
|  Chalet Girl (2011)|
|  Chosen, The (1981)|
|Color of Paradise...|
|For All Mankind (...|
|I Know Where I'm ...|
|In the Realms of ...|
|Innocents, The (1...|
|Mutiny on the Bou...|
|      Niagara (1953)|
|Parallax View, Th...|
|        Proof (1991)|
|Road Home, The (W...|
|Roaring Twenties,...|
|      Scrooge (1970)|
|This Gun for Hire...|
|Twentieth Century...|
+--------------------+



successfull


In [19]:
#Que- Show the movies that have rating but no tag
query="""WITH 
    tagged_movies AS (SELECT DISTINCT movieId FROM TAGS ), 
    rated_movies AS (SELECT DISTINCT movieId FROM RATINGS), 
    t1 AS ( SELECT movieId FROM rated_movies WHERE movieId NOT IN (SELECT movieId FROM tagged_movies))
    SELECT m.title
    FROM MOVIES m
    JOIN t1 ON m.movieId = t1.movieId
    ORDER BY m.title;"""

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimitor',',').save('/tmp/output/assign2/rating_NotTag.csv')
print("successfull")

+--------------------+
|               title|
+--------------------+
|          '71 (2014)|
|'Hellboy': The Se...|
|'Round Midnight (...|
| 'Salem's Lot (2004)|
|'Til There Was Yo...|
|'Tis the Season f...|
|  'burbs, The (1989)|
|'night Mother (1986)|
|*batteries not in...|
|...All the Marble...|
|00 Schneider - Ja...|
|   1-900 (06) (1994)|
|           10 (1979)|
|10 Cent Pistol (2...|
|10 Items or Less ...|
|     10 Years (2011)|
|    10,000 BC (2008)|
|    100 Girls (2000)|
|  100 Streets (2016)|
|101 Dalmatians II...|
+--------------------+
only showing top 20 rows



successfull


In [22]:
#Que - Focusing on the rated untagged movies with more than 30 user ratings,
#show the top 10 movies in terms of average rating and number of ratings

query="""SELECT 
    m.title,
    COUNT(*) AS num_ratings,
    AVG(r.rating) AS avg_rating
    FROM RATINGS r
    JOIN MOVIES m ON r.movieId = m.movieId
    LEFT JOIN TAGS t ON r.movieId = t.movieId
    WHERE t.movieId IS NULL
    GROUP BY m.title
    HAVING COUNT(*) > 30
    ORDER BY num_ratings desc
    LIMIT 10;"""

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimitor',',').save('/tmp/output/assign2/top_10_avgratings&count_ratings.csv')
print("successfull")

+--------------------+-----------+------------------+
|               title|num_ratings|        avg_rating|
+--------------------+-----------+------------------+
|American Beauty (...|        204| 4.056372549019608|
|Ace Ventura: Pet ...|        161| 3.040372670807453|
|    Mask, The (1994)|        157|3.1847133757961785|
|     Die Hard (1988)|        145|3.8620689655172415|
|Die Hard: With a ...|        144|3.5555555555555554|
|Groundhog Day (1993)|        143| 3.944055944055944|
|Dumb & Dumber (Du...|        133|3.0601503759398496|
|    GoldenEye (1995)|        132| 3.496212121212121|
|Monsters, Inc. (2...|        132| 3.871212121212121|
|Austin Powers: Th...|        121|3.1983471074380163|
+--------------------+-----------+------------------+



successfull


In [30]:
#What is the average number of tags per movie in tagsDF? And the
#average number of tags per user? How does it compare with the
#average number of tags a user assigns to a movie?

query="""WITH avgTag_perMovie AS (
    SELECT AVG(tag_count) AS avg_tag_perMovie
    FROM (
        SELECT movieID, COUNT(*) AS tag_count
        FROM TAGS
        GROUP BY movieID
    ) AS movie_tags
),
avgTag_perUser AS (
    SELECT AVG(tag_count) AS avg_tag_perUser
    FROM (
        SELECT userID, COUNT(*) AS tag_count
        FROM TAGS
        GROUP BY userID
    ) AS user_tags
)
SELECT 
    m.avg_tag_perMovie,
    u.avg_tag_perUser,
    CASE 
        WHEN u.avg_tag_perUser > m.avg_tag_perMovie THEN 'Tags per user is higher'
        ELSE 'Tags per movie is higher'
    END AS comparison
FROM avgTag_perMovie m
CROSS JOIN avgTag_perUser u;"""

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/tags_per_movieVStags_per_user.csv')

+------------------+---------------+--------------------+
|  avg_tag_perMovie|avg_tag_perUser|          comparison|
+------------------+---------------+--------------------+
|2.3428753180661577|           63.5|Tags per user is ...|
+------------------+---------------+--------------------+



In [36]:
#Identify the users that tagged movies without rating them
query="""SELECT DISTINCT T.userID 
        FROM TAGS T  LEFT JOIN RATINGS R
            ON T.movieID = R.movieID 
        WHERE R.userID IS NULL;
       """

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/user_taged_but_Notrate.csv')

+------+
|userID|
+------+
|   474|
|   288|
|   543|
|   318|
+------+



In [42]:
#What is the average number of ratings per user in ratings DF? And the
#average number of ratings per movie?
query=""" WITH 
        AVGrating_perUser as(Select AVG(rating_count)as avg_rating_perUser From (SELECT userID, count(*) as rating_count
            FROM RATINGS group by userID)),
        AVGrating_perMovie as(Select AVG(rating_count)as avg_rating_perMovie From (SELECT movieID, count(*) as rating_count
            FROM RATINGS group by movieID))
         
        Select U.avg_rating_perUser ,M.avg_rating_perMovie from AVGrating_perUser U CROSS JOIN AVGrating_perMovie M;
       """

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/rating_per_movieVSrating_per_user.csv')

+------------------+-------------------+
|avg_rating_perUser|avg_rating_perMovie|
+------------------+-------------------+
|165.30491803278687| 10.369806663924312|
+------------------+-------------------+



In [44]:
#What is the predominant (frequency based) genre per rating level?
query=""" WITH genre_counts AS (
    SELECT
        r.rating,
        m.genres,
        COUNT(*) AS genre_count,
        DENSE_RANK() OVER (
            PARTITION BY r.rating
            ORDER BY COUNT(*) DESC
        ) AS genre_rank
    FROM RATINGS r
    LEFT JOIN MOVIES m
        ON r.movieID = m.movieID
    GROUP BY
        r.rating,
        m.genres
)

SELECT
    rating,
    genres AS most_frequent_genre
FROM genre_counts
WHERE genre_rank = 1
ORDER BY rating DESC;
       """

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/predominant_genre_perRating.csv')

+------+-------------------+
|rating|most_frequent_genre|
+------+-------------------+
|   5.0|              Drama|
|   4.5|              Drama|
|   4.0|              Drama|
|   3.5|             Comedy|
|   3.0|             Comedy|
|   2.5|             Comedy|
|   2.0|             Comedy|
|   1.5|             Comedy|
|   1.0|             Comedy|
|   0.5|             Comedy|
+------+-------------------+



In [48]:
#What is the predominant tag per genre and the most tagged genres?
query=""" WITH genre_counts AS (
    SELECT
        t.tag,
        m.genres,
        COUNT(*) AS genre_count,
        DENSE_RANK() OVER (
            PARTITION BY t.tag
            ORDER BY COUNT(*) DESC
        ) AS genre_rank
    FROM TAGS t
    LEFT JOIN MOVIES m
        ON t.movieID = m.movieID
    GROUP BY
        t.tag,
        m.genres
)

SELECT
    genres,
    tag AS most_frequent_tag
FROM genre_counts
WHERE genre_rank = 1
ORDER BY genres DESC;
       """

output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/predominant_tag_pergenre.csv')

+---------------+------------------+
|         genres| most_frequent_tag|
+---------------+------------------+
|        Western|     Coen Brothers|
|        Western|          Dialogue|
|        Western|      Jeff Bridges|
|        Western|      Kurt Russell|
|        Western|        Matt Damon|
|        Western|        characters|
|        Western| Samuel L. Jackson|
|        Western|           Western|
|        Western|             humor|
|        Western|       predictable|
|        Western|  tension building|
|        Western|           violent|
|       Thriller|    Juliette Lewis|
|       Thriller|   Martin Scorsese|
|       Thriller|               bad|
|       Thriller|based on a TV show|
|       Thriller|            horror|
|       Thriller|           intense|
|       Thriller|       transplants|
|Sci-Fi|Thriller|              good|
+---------------+------------------+
only showing top 20 rows



In [55]:
#What are the most predominant (popularity based) movies?
query="""
    SELECT
    m.title,
    COUNT(DISTINCT r.userID) AS user_count
        FROM MOVIES m JOIN RATINGS r
    ON m.movieID = r.movieID
    GROUP BY  m.movieID, m.title
    ORDER BY user_count DESC
        LIMIT 10;
    """


output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/Most_predominant_movie.csv')

+--------------------+----------+
|               title|user_count|
+--------------------+----------+
| Forrest Gump (1994)|       329|
|Shawshank Redempt...|       317|
| Pulp Fiction (1994)|       307|
|Silence of the La...|       279|
|  Matrix, The (1999)|       278|
|Star Wars: Episod...|       251|
|Jurassic Park (1993)|       238|
|   Braveheart (1995)|       237|
|Terminator 2: Jud...|       224|
|Schindler's List ...|       220|
+--------------------+----------+



In [57]:
#Top 10 movies in terms of average rating (provided more than 30 users reviewed them)

query="""
    SELECT
    m.title,
    ROUND(t.avg_rating, 9) AS avg_rating
    FROM (
    SELECT
        movieID,
        AVG(rating) AS avg_rating
        FROM RATINGS
        GROUP BY movieID
        HAVING COUNT(DISTINCT userID) > 30
        ) t
    JOIN MOVIES m ON t.movieID = m.movieID
    ORDER BY t.avg_rating DESC
    LIMIT 10;
    """


output= spark.sql(query)
output.show()

# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format("csv").option('header','true').option('delimiter',',').save('/tmp/output/assign2/TOP10_movie_MoreThan30User.csv')


+--------------------+-----------+
|               title| avg_rating|
+--------------------+-----------+
|Shawshank Redempt...|4.429022082|
|Lawrence of Arabi...|        4.3|
|Godfather, The (1...|  4.2890625|
|   Fight Club (1999)| 4.27293578|
|Cool Hand Luke (1...|4.271929825|
|Dr. Strangelove o...|4.268041237|
|  Rear Window (1954)|4.261904762|
|Godfather: Part I...|4.259689922|
|Departed, The (2006)|4.252336449|
|   Goodfellas (1990)|       4.25|
+--------------------+-----------+

